# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### Record Sets, Fields, and Columns
Each entity (record set, field, column) is uniquely referenced by its `@id`.

In [ ]:
# Get available record sets and their @id
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    record_set_ids.append(record_set['@id'])
    print(f"Record Set @id: {record_set['@id']}, name: {record_set.get('name', 'N/A')}")

# Show fields for each record set
for record_set in dataset.metadata.record_sets:
    print(f"\nFields for Record Set {record_set['@id']}:")
    for field in record_set.get('fields', []):
        print(f"  Field @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
        if 'column' in field:
            print(f"    Column @id: {field['column']['@id']} | column name: {field['column'].get('name', 'N/A')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id` values identified above.

In [ ]:
# Extract data from all available record sets
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for record set {record_set_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping by key attributes.

**Note:** Entities are referenced by their `@id` for consistency.

In [ ]:
# Example: Analyze the main tabular record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]

    # Find a numeric field @id
    numeric_field_id = None
    group_field_id = None
    for record_set in dataset.metadata.record_sets:
        if record_set['@id'] == main_record_set_id:
            for field in record_set.get('fields', []):
                if field.get('dataType', '').lower() in ['integer', 'float', 'number']:
                    numeric_field_id = field['@id']
                elif field.get('dataType', '').lower() in ['text', 'string']:
                    group_field_id = field['@id']
            break

    print(f"Numeric Field @id: {numeric_field_id}")
    print(f"Group Field @id: {group_field_id}")

    if numeric_field_id and numeric_field_id in df.columns:
        # Filter records where numeric_field > 10
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped by {group_field_id}, mean of {numeric_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib to visualize the numeric field distribution and group-wise statistics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id and (numeric_field_id in df.columns):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Grouped boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploration of the FAIR^2 colorectal cancer dataset using `mlcroissant`.

- Entities were referenced by their `@id` for reproducibility.
- Record sets, fields, and columns were inspected from the Croissant schema.
- Numeric fields were filtered, normalized, and grouped for analysis.
- Visualizations were provided for distribution and group comparison.

Further analysis can be extended by investigating other record sets or fields, adding more domain-specific processing, or integrating the data with additional sources.